Authenticate user <br>
Read Email <br>
Sents emails with HITL ( Human in the loop)<br>

s

In [22]:
from dotenv import load_dotenv
from langchain.agents import create_agent, AgentState
from langchain.messages import ToolMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain.agents.middleware import HumanInTheLoopMiddleware, wrap_model_call, ModelRequest, ModelResponse
from typing import Callable
from langchain.tools import tool, ToolRuntime

load_dotenv()

True

In [23]:
from dataclasses import dataclass

@dataclass
class PrivilegedUser:
    email: str = "berry_jhonson@gmail.com"
    password: str = "rockyou!"

class AuthenticatedState(AgentState):
    authenticated: bool

In [61]:
@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email, from given address."""
    return "Hello, i hope ur happy about our meeting!"

@tool
def send_email(destination:str, body: str) -> str:
    """Send and email."""
    return f"Email sent to {destination} with body of {body}!"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate user with given email address and password."""
    if email == runtime.context.email and password == runtime.context.password:
        return Command(
            update={
                "authenticated": True,
                "messages": [
                    ToolMessage("User Authenticated!", tool_call_id=runtime.tool_call_id)
                ]
            }
        )
    else:
        return Command(
            update={
                "authenticated": False,
                "messages": [
                    ToolMessage("Authentication failed!", tool_call_id=runtime.tool_call_id)
                ]
            }
        )

In [67]:
@wrap_model_call
def dynamic_tool_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Allow to read and write email, only if user provides correct email address and password."""
    authenticated = request.state.get("authenticated")
    if authenticated:
        tools = [read_email, send_email]
    else:
        tools = [authenticate]
    request = request.override(tools=tools)

    return handler(request)

In [68]:
from langchain.agents.middleware import dynamic_prompt
@dynamic_prompt
def dynaminc_prompt_func(request: ModelRequest) -> str:
    """ Generate system prompt based on authentication status."""
    authenticated = request.state.get("authenticated")
    if authenticated:
        return "You are helpfull assistant that can check and read emails for authenticated user."
    else:
        return "You are helpfull assistant that let's user authenticate himself."

In [69]:
model = "gpt-5-nano"
config = {"configurable": {"thread_id": "1"}}

agent = create_agent(
    model = model,
    tools=[read_email, send_email, authenticate],
    state_schema=AuthenticatedState,
    context_schema=PrivilegedUser,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
                "authenticate": False
            },
            description_prefix="Tool execution requires approval."
        ),
        dynamic_tool_call
    ]
)

In [70]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Hello i wanted to check my email box.")]},
    context=PrivilegedUser,
    config=config
)
print(response["messages"][-1].content)

Sure, I can help you check your inbox. Please provide your email and password to sign in.

- Email:
- Password:

Note: Share credentials only here for this session. I won’t store them beyond signing you in. If you’d rather not share your password, I can guide you through a password reset or help you access your mail via your provider’s site.


In [71]:
response = agent.invoke(
    {"messages": [HumanMessage(content="my address is berry_johnson@gmail.com and my password is rockyou!.")]},
    context=PrivilegedUser,
    config=config
)
print(response["messages"][-1].content)

Authentication failed. Here are some options:

- Try signing in again with the correct password: re-enter your password carefully (watch for spaces or caps).
- If you don’t remember the password, reset it:
  - Go to Google's sign-in recovery: https://accounts.google.com/signin/recovery
  - Enter your email (berry_johnson@gmail.com) and follow the prompts. You may need to use a recovery phone number or email.
- If you suspect account security issues, check Google’s security alerts and consider changing your password from a trusted device.

Would you like me to guide you step-by-step through the Google account recovery, or would you prefer to try signing in again with a new password? If you want me to retry, please provide the password you want me to use (or confirm you want to reset).


In [72]:
"berry_jhonson@gmail.com"

'berry_jhonson@gmail.com'

In [73]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Sorry, my address is berry_jhonson@gmail.com")]},
    context=PrivilegedUser,
    config=config
)
print(response["messages"][-1].content)

You're signed in as berry_jhonson@gmail.com. I can help you check your inbox. Here are a few options:
- Quick look at latest messages
- Read specific email by subject or sender
- Search for emails with keywords
- Manage labels or archive/delete

Great! What would you like to do first? Also, please confirm if you want me to operate only within this session and not access anything outside your current inbox. Note: I won't access any emails unless you instruct me to. You can trust me to keep your data private.


In [ ]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Check last message")]},
    context=PrivilegedUser,
    config=config
)
print(response["messages"][-1].content)

C:\Users\safan\Documents\GitHub\lca-lc-foundations\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=<class '__main__.PrivilegedUser'>, input_type=type])
  return self.__pydantic_serializer__.to_python(
